In [2]:
!pip install torch matplotlib

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [4]:
class PINN(nn.Module):
    def __init__(self, in_dim=2, out_dim=1, hidden_dim=64, n_hidden=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z, xi):
        # z, xi: tensors of shape (N, 1)
        inp = torch.cat([z, xi], dim=1)
        return self.net(inp)


In [5]:
def pinn_residual(model, z, xi):
    """
    Compute residual of ODE:
    x'' + 2*xi*x' + x = 0
    """
    z = z.clone().detach().requires_grad_(True)
    xi = xi.clone().detach().requires_grad_(True)

    x = model(z, xi)                 # (N, 1)
    dx_dz = torch.autograd.grad(
        x, z,
        grad_outputs=torch.ones_like(x),
        create_graph=True,
        retain_graph=True
    )[0]

    d2x_dz2 = torch.autograd.grad(
        dx_dz, z,
        grad_outputs=torch.ones_like(dx_dz),
        create_graph=True,
        retain_graph=True
    )[0]

    res = d2x_dz2 + 2.0 * xi * dx_dz + x
    return res


In [6]:
x0 = 0.7
v0 = 1.2

def ic_loss(model, n_ic=64):
    # Sample xi in [0.1, 0.4] for initial conditions
    xi_ic = torch.rand(n_ic, 1, device=device) * (0.4 - 0.1) + 0.1
    z_ic = torch.zeros_like(xi_ic, device=device)

    z_ic.requires_grad_(True)
    xi_ic.requires_grad_(True)

    x = model(z_ic, xi_ic)
    dx_dz = torch.autograd.grad(
        x, z_ic,
        grad_outputs=torch.ones_like(x),
        create_graph=True,
        retain_graph=True
    )[0]

    loss_x0 = torch.mean((x - x0) ** 2)
    loss_v0 = torch.mean((dx_dz - v0) ** 2)
    return loss_x0 + loss_v0


In [7]:
def physics_loss(model, n_collocation=1024):
    # Sample z in [0, 20], xi in [0.1, 0.4]
    z = torch.rand(n_collocation, 1, device=device) * 20.0
    xi = torch.rand(n_collocation, 1, device=device) * (0.4 - 0.1) + 0.1

    res = pinn_residual(model, z, xi)
    return torch.mean(res**2)


In [ ]:
model = PINN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 5000

for epoch in range(1, n_epochs + 1):
    optimizer.zero_grad()

    loss_pde = physics_loss(model, n_collocation=1024)
    loss_ic = ic_loss(model, n_ic=128)

    loss = loss_pde + loss_ic

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Total Loss: {loss.item():.4e}, PDE: {loss_pde.item():.4e}, IC: {loss_ic.item():.4e}")


Epoch 500, Total Loss: 8.2844e-03, PDE: 8.2682e-03, IC: 1.6199e-05
Epoch 1000, Total Loss: 6.2434e-03, PDE: 5.8762e-03, IC: 3.6721e-04
Epoch 1500, Total Loss: 3.9367e-03, PDE: 3.9065e-03, IC: 3.0170e-05
Epoch 2000, Total Loss: 2.8859e-03, PDE: 2.7897e-03, IC: 9.6235e-05
Epoch 2500, Total Loss: 3.3760e-03, PDE: 3.0575e-03, IC: 3.1851e-04
Epoch 3000, Total Loss: 2.6018e-03, PDE: 2.4068e-03, IC: 1.9502e-04


In [ ]:
model.eval()

def predict_solution(xi_value, n_points=200):
    z = torch.linspace(0.0, 20.0, n_points, device=device).view(-1, 1)
    xi = torch.full_like(z, xi_value, device=device)
    with torch.no_grad():
        x = model(z, xi)
    return z.cpu().numpy().flatten(), x.cpu().numpy().flatten()

xis_to_plot = [0.1, 0.2, 0.3, 0.4]

plt.figure(figsize=(8, 5))
for xi_val in xis_to_plot:
    z_vals, x_vals = predict_solution(xi_val)
    plt.plot(z_vals, x_vals, label=f"xi={xi_val}")

plt.xlabel("z")
plt.ylabel("x(z)")
plt.title("Damped harmonic oscillator PINN solutions")
plt.legend()
plt.grid(True)
plt.show()
